In [58]:
import warnings
warnings.filterwarnings('ignore')

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process

In [60]:
url = "https://www.nytimes.com/newsgraphics/polls/house.csv"
polls = pd.read_csv(url)
polls.to_csv('../../2026_data/polls/house.csv')
polls = polls[
    (polls['state'] != 'US') &
    (polls['stage'] == 'general')
]
polls.head()

,poll_id,pollster_id,pollster,sponsor_ids,sponsors,display_name,pollster_rating_id,pollster_rating_name,numeric_grade,pollscore,methodology,transparency_score,state,start_date,end_date,sponsor_candidate_id,sponsor_candidate,sponsor_candidate_party,question_id,sample_size,population,subpopulation,population_full,tracking,created_at,notes,url,url_article,url_topline,url_crosstab,source,internal,partisan,cycle,office_type,seat_name,seat_number,election_date,stage,party,pct,answer,candidate_name,candidate_id,race_id,ranked_choice_round,ranked_choice_reallocated,ranked_choice_final,nationwide_match,hypothetical
2,1f6ed24d-8b93-4444-a8bc-f39f1c65f819,c8e8dc89-672f-4b1c-8958-669cd59b35c2,DCCC Targeting Team,NaN,NaN,DCCC Targeting Team,68.0,DCCC Targeting and Analytics Department,NaN,NaN,Live Phone/Text-to-Web,NaN,ME,8/11/26,8/13/26,NaN,NaN,NaN,759998b6-19b4-41d8-9581-cd63c8762c14,621.0,lv,NaN,lv,NaN,9/22/26 10:23,NaN,https://dccc.org/%F0%9F%9A%A8-new-polls-dead-h...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Second Congressional District,2.0,2026-11-03,general,DEM,48.0,Dunlap,Matt Dunlap,cd0278bb-3af0-435a-9abd-8ec4471c32df,c587aa21-f06f-47d6-ae64-ebbd9d0a43d7,NaN,False,False,NaN,NaN
3,1f6ed24d-8b93-4444-a8bc-f39f1c65f819,c8e8dc89-672f-4b1c-8958-669cd59b35c2,DCCC Targeting Team,NaN,NaN,DCCC Targeting Team,68.0,DCCC Targeting and Analytics Department,NaN,NaN,Live Phone/Text-to-Web,NaN,ME,8/11/26,8/13/26,NaN,NaN,NaN,759998b6-19b4-41d8-9581-cd63c8762c14,621.0,lv,NaN,lv,NaN,9/22/26 10:23,NaN,https://dccc.org/%F0%9F%9A%A8-new-polls-dead-h...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Second Congressional District,2.0,2026-11-03,general,REP,49.0,LePage,Paul LePage,94436f81-105d-46a1-a241-ed6f9c76a03b,c587aa21-f06f-47d6-ae64-ebbd9d0a43d7,NaN,False,False,NaN,NaN
18,ed869046-fc8d-4365-98b2-373d6cc882ba,c8e8dc89-672f-4b1c-8958-669cd59b35c2,DCCC Targeting Team,NaN,NaN,DCCC Targeting Team,68.0,DCCC Targeting and Analytics Department,NaN,NaN,NaN,NaN,VA,9/15/26,9/17/26,NaN,NaN,NaN,36970122-c545-4d6f-bfa8-c7eb074c70f4,519.0,lv,NaN,lv,NaN,9/21/26 14:48,NaN,https://www.virginiascope.com/internal-poll-sh...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,First Congressional District,1.0,2026-11-03,general,NONE,7.0,Don't know,Don't know,d8390906-54c4-4050-9005-4934a65fa7b1,b4ed2906-ddb9-4b3a-880e-b8443d5dcc16,NaN,False,False,NaN,NaN
19,ed869046-fc8d-4365-98b2-373d6cc882ba,c8e8dc89-672f-4b1c-8958-669cd59b35c2,DCCC Targeting Team,NaN,NaN,DCCC Targeting Team,68.0,DCCC Targeting and Analytics Department,NaN,NaN,NaN,NaN,VA,9/15/26,9/17/26,NaN,NaN,NaN,36970122-c545-4d6f-bfa8-c7eb074c70f4,519.0,lv,NaN,lv,NaN,9/21/26 14:48,NaN,https://www.virginiascope.com/internal-poll-sh...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,First Congressional District,1.0,2026-11-03,general,DEM,46.0,Taylor (Shannon),Shannon Taylor,328f958d-6533-4b4a-bb15-8af744a8abd5,b4ed2906-ddb9-4b3a-880e-b8443d5dcc16,NaN,False,False,NaN,NaN
20,ed869046-fc8d-4365-98b2-373d6cc882ba,c8e8dc89-672f-4b1c-8958-669cd59b35c2,DCCC Targeting Team,NaN,NaN,DCCC Targeting Team,68.0,DCCC Targeting and Analytics Department,NaN,NaN,NaN,NaN,VA,9/15/26,9/17/26,NaN,NaN,NaN,36970122-c545-4d6f-bfa8-c7eb074c70f4,519.0,lv,NaN,lv,NaN,9/21/26 14:48,NaN,https://www.virginiascope.com/internal-poll-sh...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,First Congressional District,1.0,2026-11-03,general,REP,47.0,Wittman,Robert Wittman,ef7eb95e-2607-408d-9cd8-b986319a83b5,b4ed2906-ddb9-4b3a-880e-b8443d5dcc16,NaN,False,False,NaN,NaN


In [61]:
pd.set_option('display.max_columns', None)

In [62]:
polls = polls[~((polls['state'] == 'MO') & (pd.to_datetime(polls['start_date']) < pd.to_datetime("2026-09-03")))]

In [63]:
polls_pivot = pd.pivot_table(data=polls, values='pct', index=['poll_id', 'question_id', 'state', 'seat_number'], columns='candidate_name', aggfunc='first')
polls_pivot = polls_pivot.reset_index()
polls_pivot.head()

candidate_name,poll_id,question_id,state,seat_number,Aaron Flint,Adam Ortiz,Alex Kelloff,Amanda Green,Ammar Campa-Najjar,Andy Harris,Andy Ogles,Angela Gonzales-Torres,Anna Paulina Luna,Anthony Constantino,Ashley Bell,Austin Ahlman,Bajun Mavalwalla,Bale Dalton,Becca Balint,Belinda Keiser,Bill Hill,Bill Huizenga,Bill Reeside,Blake Gendebien,Bob Brooks,Bob Harvie,Bobby Pulido,Brad Finstad,Brad Smith,Brandon Herrera,Brendan Gomez,Brennan Barrington,Brent Taylor,Brian Fitzpatrick,Brian Miller,Brian Poindexter,Brice Barnes,Bridget Brink,Brinker Harding,Bryan Steil,Cait Conley,Carlos De La Cruz,Carmela Conroy,Casey Askar,Charlie Hatcher,Chaz Molder,Cherlynn Stevenson,Chris Backemeyer,Chris Gallant,Chris Getty,Chris Harden,Chris Jones,Chris Royal,Chris Schmidt,Christina Bohannan,Christina Hines,Clay Strickland,Connie Chan,Cory Mills,Curtis Goodwin,Dan Green,Dan Schwartz,Darrell Issa,Darren Soto,David Flippo,David G. Valadao,David Gedert,David Rouzer,David Womack,Denise Powell,Derek Merrin,Derrick Van Orden,Don Davis,Don Leonard,Don't know,Don't know/Someone else,Don't know/Would not vote,Doris Matsui,Dwayne Romero,Ed Hershey,Eddie Espinoza,Elaine G. Luria,Eli Crane,Eliott Rodriguez,Eric Chung,Eric Conroy,Eric Flores,Eric Hafner,Eric Pratt,Esther Kim-Varet,French Hill,Gabe Vasquez,Generic Democrat,Generic Libertarian,Generic Republican,George Moraitis,Gerald Malloy,Gina Swoboda,Glenn Grothman,Greg Cunningham,Greg Landsman,Greg Murphy,Henry Cuellar,J.D. Ford,Jackie Auringer,Jake Johnson,James Johnson,James Pericola,Jamie Ager,Janelle Stelson,Jared Golden,Jay Feely,Jeff Crank,Jeff Hurd,Jeff Van Drew,Jen Kiggans,Jennifer Jenkins,Jennifer Konfrst,Jenny Costa Honeycutt,Jessica Killin,Jim C. McDermott,Jim Desmond,Jimmy Gomez,JoAnna Mendoza,Joe Baldacci,Joe Mitchell,Joe Strada,John Braun,John Cavanaugh,John Cowan,John McGuire,John Vincent,John Williams,Johnny Garcia,Jonathan Nez,Jordan Wood,Juan Ciscomani,Justin Pearson,Kaela Berg,Kathy Castor,Katy Padilla Stout,Kaylee Peterson,Keith Gross,Kelly Kirschner,Ken Calvert,Kevin Kiley,Kevin Steele,Kimberly Hardy,Kristina Knickerbocker,LaKesha Womack,Laurie Buckhout,Leela Gray,Lily Tang Williams,Lindsay James,Lupe Castillo,Lynn Chapman,Maad Abu-Ghazalah,Mac Deford,Maggie Goodlander,Mai Vang,Marcy Kaptur,Margo Ellis,Mariannette Miller-Meeks,Marie Gluesenkamp Perez,Mark Coester,Mark Smith,Marlon Duran,Marni von Wilpert,Mary Peltola,María Elvira Salazar,Matt Cavanaugh,Matt Dunlap,Matt Klein,Matt Little,Matt Maasdam,Matt Rains,Matt Schultz,Max Miller,Michael Baumgartner,Michael Bridgford,Michael Eisenhauer,Michael Thurow,Micheál O'Leary,Mike Beltran,Mike Bouchard,Mike Carey,Mike Flood,Mike Haridopolos,Mike Lawler,Mike Turner,Mitchell Berman,Monica De La Cruz,Nancy Lacore,Nancy Pelosi,Nate Powell,Neither,Nicholas J. LaLota,Nicholas Zateslo,Nick Begich,Nick Sheedy,Oliver Larkin,Other named candidates,Paige Cognetti,Pat Harrigan,Pat Ryan,Patrick McCracken,Patty Garcia,Paul LePage,Pia Dandiya,Ralph Alvarado,Randy Villegas,Raymond Smith Jr.,Rebecca Bennett,Rebecca Cooke,Rhett Marques,Rich McCormick,Richard Hudson,Richard Ojeda,Richard Pan,Rob Bresnahan Jr.,Robert Wittman,Russ Fulcher,Russell Fry,Ryan Busse,Ryan Dotson,Ryan E. Mackenzie,Ryan Zinke,Saikat Chakrabarti,Sam Forstag,Sarah Trone Garriott,Sarah Zabel,Scott Perry,Scott Singer,Scott Wiener,Seamus O'Toole,Sean McCann,Shannon Taylor,Shomari Figures,Someone else,Sydney Gruters,Tano Tijerina,Teresa Benitez-Thompson,Thomas Aaron Bailey,Thomas H. Kean Jr.,Thomas McMasters,Tim Greimel,Tim Moore,Tim Sheehy,Tina Shah,Tom Barrett,Tom Perriello,Tony D’Arrigo,Tony Kozycki,Troy Downing,Vicente Gonzalez,Victoria Spartz,William Lawrence,Would not vote,Yen Bailey,Young Kim,Zach Dembo,Zach Nunn
0,016369fb-4a26-42ad-a583-39067289cab2,65c0d75b-5121-4107-b61a-95b2534fab6e,WI,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [64]:
polls_pivot.shape

(294, 262)

In [65]:
candinfo_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQZC7cVru6ltLR2e8XN5jdJPfxfj42BAxUApe3Zq3_ENQjLtYntmAxD0pIHqEUJ4ZFLXlybKJdkLf2r/pub?output=csv'
candinfo = pd.read_csv(candinfo_url)

In [66]:
dem_uncont = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_uncont = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]
candinfo['state_po'] = candinfo['cd'].map(lambda x: x[:2]).astype(str)
candinfo['district_number'] = candinfo['cd'].map(lambda x: 0 if x[3:] == 'AL' else int(x[3:]))
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1
2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4


In [67]:
dem_uncont.shape, rep_uncont.shape

((16, 5), (1, 5))

In [68]:
polled_cands = polls_pivot.columns.values[4:]
running_dems = np.unique(candinfo['dem_cand'].astype(str))
running_reps = np.unique(candinfo['rep_cand'].astype(str))
all_running_cands = np.concatenate([running_dems, running_reps])

In [69]:
hypo_cands = []
for c in polled_cands:
    fuzzymatch = process.extractOne(c, all_running_cands, scorer=fuzz.token_sort_ratio, score_cutoff=80)
    if fuzzymatch is None:
        hypo_cands.append(c)

In [70]:
for h in hypo_cands:
    if h in ['Nicholas J. LaLota', 'Robert J. Wittman', 'Thomas H. Kean',
 'Thomas H. Kean Jr.', "Don't know",
 "Don't know/Someone else",
 "Don't know/Would not vote", 'Someone else', 'Would not vote']:
        continue
    # print(h, polls_pivot["Christina Bohannan"].isna().all())
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

for h in ['John Williams']: # names that weren't caught in hypo_cands but should've been
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

In [71]:
polls_pivot.columns.values

array(['poll_id', 'question_id', 'state', 'seat_number', 'Aaron Flint',
       'Amanda Green', 'Andy Harris', 'Anna Paulina Luna',
       'Anthony Constantino', 'Ashley Bell', 'Bale Dalton',
       'Becca Balint', 'Bill Hill', 'Bill Huizenga', 'Blake Gendebien',
       'Bob Brooks', 'Bob Harvie', 'Bobby Pulido', 'Brad Finstad',
       'Brad Smith', 'Brandon Herrera', 'Brent Taylor',
       'Brian Fitzpatrick', 'Brian Miller', 'Brian Poindexter',
       'Brinker Harding', 'Bryan Steil', 'Cait Conley',
       'Carlos De La Cruz', 'Carmela Conroy', 'Casey Askar',
       'Charlie Hatcher', 'Chaz Molder', 'Chris Backemeyer',
       'Chris Gallant', 'Chris Harden', 'Chris Jones', 'Chris Schmidt',
       'Christina Bohannan', 'Christina Hines', 'Dan Green',
       'Dan Schwartz', 'Darren Soto', 'David Flippo', 'David G. Valadao',
       'David Rouzer', 'Denise Powell', 'Derek Merrin',
       'Derrick Van Orden', 'Don Davis', 'Don Leonard', "Don't know",
       "Don't know/Someone else", "Don'

In [72]:
polls_pivot.shape

(140, 177)

In [73]:
rel_polls = polls[(polls['poll_id'].isin(np.unique(polls_pivot['poll_id']))) &
    (polls['question_id'].isin(np.unique(polls_pivot['question_id'])))]

In [74]:
np.unique(rel_polls['candidate_name'])

array(['Aaron Flint', 'Andy Harris', 'Anna Paulina Luna',
       'Anthony Constantino', 'Ashley Bell', 'Bill Hill', 'Bill Huizenga',
       'Blake Gendebien', 'Bob Brooks', 'Bob Harvie', 'Bobby Pulido',
       'Brad Finstad', 'Brad Smith', 'Brandon Herrera', 'Brent Taylor',
       'Brian Fitzpatrick', 'Brian Poindexter', 'Brinker Harding',
       'Bryan Steil', 'Cait Conley', 'Carlos De La Cruz',
       'Carmela Conroy', 'Casey Askar', 'Chris Backemeyer',
       'Chris Gallant', 'Chris Harden', 'Chris Jones', 'Chris Schmidt',
       'Christina Bohannan', 'Christina Hines', 'Dan Green',
       'Dan Schwartz', 'Darren Soto', 'David Flippo', 'David G. Valadao',
       'David Rouzer', 'Denise Powell', 'Derrick Van Orden', 'Don Davis',
       "Don't know", "Don't know/Would not vote", 'Dwayne Romero',
       'Elaine G. Luria', 'Eli Crane', 'Eliott Rodriguez', 'Eric Chung',
       'Eric Conroy', 'Eric Flores', 'Eric Pratt', 'French Hill',
       'Gabe Vasquez', 'Glenn Grothman', 'Greg Cunnin

In [75]:
rel_polls.shape

(372, 50)

In [76]:
rel_polls.to_csv('transformed/relevent_house_polls.csv')